<a href="https://colab.research.google.com/github/catrina-llamas-1/Cats-Repository/blob/main/amenities_matcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Amenities Matcher: Dataset A ↔ Dataset B (with Fuzzy Matching)

This notebook:
1. Loads Dataset A and Dataset B
2. Normalises amenity strings and **fuzzy-matches** them against a canonical list
3. **Fuzzy-matches** rows across datasets by address (handles abbreviation differences)
4. Writes matched canonical amenities back into Dataset A’s `amenities` column
5. Exports the updated Dataset A (retaining its original format)

**Fuzzy matching** handles:
- Minor spelling variants (`Bio-Tech/ Lab Space` → `Bio-Tech / Lab Space`)
- Case differences (`fitness center` → `Fitness Center`)
- Partial abbreviation differences in addresses (`Ave NW` → `Avenue Northwest`)
- Whitespace / punctuation inconsistencies

## 0. Configuration — Edit This Section

In [ ]:
# ── File paths ───────────────────────────────────────────────────────────────────
DATASET_A_PATH = "PI Whyte Ave.xlsx"      # Supports .xlsx, .xls, or .csv
DATASET_B_PATH = "Costar Office Dataset.xlsx"  # Supports .xlsx, .xls, or .csv
OUTPUT_PATH    = "Finished amenities updated.xlsx"  # Output will match Dataset A's format

# ── Column names (adjust if your files use different headers) ────────────────────
A_COSTAR_ID_COL      = "Costar ID"        # Costar ID column in Dataset A (primary match key)
A_ADDRESS_COL        = "Primary address"  # Address column in Dataset A
A_AMENITIES_COL      = "Amenities"        # Amenities column in Dataset A
A_PROPERTY_NAME_COL  = "Property name"    # Property name column in Dataset A (fallback)

B_COSTAR_ID_COL      = "Costar ID"        # Costar ID column in Dataset B (primary match key)
B_ADDRESS_COL        = "Property Address" # Address column in Dataset B
B_AMENITIES_COL      = "Amenities"        # Amenities column in Dataset B
B_PROPERTY_NAME_COL  = "Property Name"    # Property name column in Dataset B (fallback)

# ── Delimiter config ────────────────────────────────────────────────────────────
B_AMENITIES_DELIMITER = ","
OUTPUT_DELIMITER      = ", "

# ── Overwrite behaviour ─────────────────────────────────────────────────────────
# True  → replace any existing value in A's amenities column on a match
# False → only fill when A's amenities cell is empty / NaN
OVERWRITE_EXISTING = True

# ── Fuzzy matching thresholds (0–100; higher = stricter) ─────────────────────
# Costar ID matching is always exact — no threshold needed.

AMENITY_FUZZY_THRESHOLD = 80
# 80 → catches minor typos and punctuation differences (recommended)
# 70 → broader; may introduce false positives
# 90 → very strict; only near-identical strings match

ADDRESS_FUZZY_THRESHOLD = 90
# 90 → keeps false positives very low (recommended default)
# 85 → handles moderate abbreviation differences (Ave vs Avenue, NW vs Northwest)
# 80 → more permissive; verify fuzzy matches before trusting results

PROPERTY_NAME_FUZZY_THRESHOLD = 90
# 90 → recommended; property names are often short so a high bar avoids false matches
# 85 → slightly more permissive; useful if names have minor spelling variants

VERBOSE_FUZZY = False

## 1. Canonical Amenities List

In [8]:
CANONICAL_AMENITIES = {
    "24 Hour Access",
    "Atrium",
    "Banking",
    "Bicycle Storage",
    "Bio-Tech / Lab Space",
    "Car Charging Station",
    "Conference Center",
    "Controlled Access",
    "Courtyard",
    "Coworking Space",
    "Energy Star Labeled",
    "Fitness Center",
    "Food Service",
    "On Transit",
    "Private Terrace",
    "Property Manager on Site",
    "Restaurant",
    "Roof Terrace",
    "Security System",
    "Shower Facilities",
    "Signage",
    "Storage Space",
    "Tenant Lounge",
    "Waterfront",
}

# Explicit value conversions applied before fuzzy matching.
# Keys must be lowercase; values must be exact canonical strings.
# Add or remove entries here as your data evolves.
AMENITY_CONVERSIONS: dict[str, str] = {
    "food court":            "Food Service",
    "convenience store":     "Food Service",
    "conferencing facility": "Conference Center",
    "yard":                  "Courtyard",
    "monument signage":      "Signage",
}

print(f"Canonical list       : {len(CANONICAL_AMENITIES)} amenities.")
print(f"Explicit conversions : {len(AMENITY_CONVERSIONS)} rules.")

Canonical list       : 24 amenities.
Explicit conversions : 5 rules.


## 2. Install & Import Dependencies

In [9]:
%pip install rapidfuzz --quiet
print("rapidfuzz ready.")

rapidfuzz ready.


## 3. Helper Functions

`normalise_address` is applied to **both** datasets before any address comparison.
`normalise_name` is a lighter normalisation for property names (lowercase, remove
punctuation, collapse whitespace).

**Match priority for each Dataset A row:**
1. **Costar ID** — exact match against `b_id_lookup` (fastest, most reliable)
2. **Property address** — exact after normalisation, then fuzzy WRatio
3. **Property name** — fuzzy WRatio fallback, only when steps 1 and 2 both fail

Amenities are never transferred unless one of these three matches succeeds.

In [ ]:
import os
import re

import pandas as pd
from rapidfuzz import fuzz, process


# ── File I/O ──────────────────────────────────────────────────────────────────────

def load_file(path: str) -> pd.DataFrame:
    """Load a CSV or Excel file into a DataFrame."""
    ext = os.path.splitext(path)[-1].lower()
    if ext == ".csv":
        return pd.read_csv(path, dtype=str)
    elif ext in (".xlsx", ".xls"):
        return pd.read_excel(path, dtype=str)
    else:
        raise ValueError(f"Unsupported file type: {ext}. Use .csv, .xlsx, or .xls.")


# ── Address normalisation tables ──────────────────────────────────────────────────

_STREET_TYPES: dict[str, str] = {
    "ave":  "avenue",
    "av":   "avenue",
    "st":   "street",
    "rd":   "road",
    "blvd": "boulevard",
    "boul": "boulevard",
    "dr":   "drive",
    "ct":   "court",
    "ln":   "lane",
    "pl":   "place",
    "cr":   "crescent",
    "cres": "crescent",
    "wy":   "way",
    "hwy":  "highway",
    "pkwy": "parkway",
    "cir":  "circle",
}

_DIRECTIONS: dict[str, str] = {
    "nw": "northwest",
    "ne": "northeast",
    "sw": "southwest",
    "se": "southeast",
    "n":  "north",
    "s":  "south",
    "e":  "east",
    "w":  "west",
}

_ORDINAL_RE = re.compile(r'\b(\d+)(?:st|nd|rd|th)\b')


def normalise_address(addr) -> str:
    """
    Full address normalisation applied to BOTH datasets before any matching.

    Steps
    -----
    1. Lowercase, strip, remove punctuation (periods / commas in abbreviations)
    2. Remove ordinal suffixes  → '82nd' becomes '82', '1st' becomes '1'
    3. Expand street-type abbreviations token-by-token
    4. Expand directional abbreviations token-by-token
    5. Collapse whitespace to a single space
    """
    if pd.isna(addr):
        return ""

    text = str(addr).strip().lower()
    text = text.replace(".", "").replace(",", "")
    text = _ORDINAL_RE.sub(r"\1", text)

    tokens = text.split()
    tokens = [_STREET_TYPES.get(t, t) for t in tokens]
    text   = " ".join(tokens)

    tokens = text.split()
    tokens = [_DIRECTIONS.get(t, t) for t in tokens]
    text   = " ".join(tokens)

    return re.sub(r"\s+", " ", text).strip()


def normalise_name(name) -> str:
    """Lowercase, remove punctuation, collapse whitespace — for property name matching."""
    if pd.isna(name):
        return ""
    text = str(name).strip().lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def normalise_text(text: str) -> str:
    """Lowercase + collapse whitespace (used as fuzzy-match input for amenities)."""
    return re.sub(r"\s+", " ", str(text).strip().lower())


# ── Amenity parsing ───────────────────────────────────────────────────────────────

def parse_amenities(raw, delimiter: str) -> list[str]:
    """Split a raw amenities string into individual trimmed tokens."""
    if pd.isna(raw) or str(raw).strip() == "":
        return []
    return [a.strip() for a in str(raw).split(delimiter) if a.strip()]


# ── Fuzzy matching ────────────────────────────────────────────────────────────────

def fuzzy_match_amenity(
    raw: str,
    canonical: set,
    threshold: int,
    conversions: dict | None = None,
    verbose: bool = False,
) -> str | None:
    """
    Map a raw amenity string to the best-matching canonical amenity.

    Resolution order
    ----------------
    1. Exact match (case-sensitive)
    2. Case-insensitive exact match
    3. Explicit conversion lookup (e.g. 'Food Court' → 'Food Service')
    4. Fuzzy token_sort_ratio match above `threshold`
    """
    if raw in canonical:
        return raw

    raw_lower = raw.lower()
    for c in canonical:
        if c.lower() == raw_lower:
            if verbose:
                print(f"  [amenity ci-exact] '{raw}' → '{c}'")
            return c

    if conversions:
        converted = conversions.get(raw_lower)
        if converted is not None:
            if verbose:
                print(f"  [amenity conversion] '{raw}' → '{converted}'")
            return converted

    canonical_list  = list(canonical)
    canonical_norms = [normalise_text(c) for c in canonical_list]

    result = process.extractOne(
        normalise_text(raw),
        canonical_norms,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=threshold,
    )
    if result is not None:
        _matched_norm, score, idx = result
        canon = canonical_list[idx]
        if verbose:
            print(f"  [amenity fuzzy {score:.0f}] '{raw}' → '{canon}'")
        return canon

    return None


def fuzzy_match_address(
    addr_norm: str,
    lookup: dict,
    threshold: int,
    verbose: bool = False,
) -> str | None:
    """
    Return the lookup key that best matches `addr_norm`.

    Resolution order
    ----------------
    1. Exact key lookup (post-normalisation)
    2. rapidfuzz WRatio fuzzy match above `threshold`
    """
    if addr_norm in lookup:
        return addr_norm

    if not lookup:
        return None

    keys = list(lookup.keys())
    result = process.extractOne(
        addr_norm,
        keys,
        scorer=fuzz.WRatio,
        score_cutoff=threshold,
    )
    if result is not None:
        matched_key, score, _ = result
        if verbose:
            print(f"  [address fuzzy {score:.0f}] '{addr_norm}' → '{matched_key}'")
        return matched_key

    return None


def fuzzy_match_name(
    name_norm: str,
    lookup: dict,
    threshold: int,
    verbose: bool = False,
) -> str | None:
    """
    Return the lookup key whose normalised property name best matches `name_norm`.
    Used as a fallback when address matching produces no result.

    Resolution order
    ----------------
    1. Exact key lookup
    2. rapidfuzz WRatio fuzzy match above `threshold`
    """
    if name_norm in lookup:
        return name_norm

    if not lookup:
        return None

    keys = list(lookup.keys())
    result = process.extractOne(
        name_norm,
        keys,
        scorer=fuzz.WRatio,
        score_cutoff=threshold,
    )
    if result is not None:
        matched_key, score, _ = result
        if verbose:
            print(f"  [name fuzzy {score:.0f}] '{name_norm}' → '{matched_key}'")
        return matched_key

    return None


print("Helper functions loaded.")

In [11]:
# Verify normalisation on representative examples before running the full pipeline.
examples = [
    ("8104 82nd Ave NW",         "8104 82 avenue northwest"),
    ("8104 82 Avenue Northwest",  "8104 82 avenue northwest"),
    ("8560 Roper Rd",             "8560 roper road"),
    ("4212 Gateway Blvd NW",     "4212 gateway boulevard northwest"),
    ("10335 95th St NW",         "10335 95 street northwest"),
    ("131 1st Ave",              "131 1 avenue"),
]

all_ok = True
print(f"{'Raw input':<35}  {'Normalised':<40}  {'OK?'}")
print("-" * 85)
for raw, expected in examples:
    result = normalise_address(raw)
    ok     = result == expected
    if not ok:
        all_ok = False
    print(f"{raw:<35}  {result:<40}  {'✓' if ok else '✗  expected: ' + expected}")

print()
print("✅ All normalisation checks passed." if all_ok else "⚠️  Some checks failed — review above.")

Raw input                            Normalised                                OK?
-------------------------------------------------------------------------------------
8104 82nd Ave NW                     8104 82 avenue northwest                  ✓
8104 82 Avenue Northwest             8104 82 avenue northwest                  ✓
8560 Roper Rd                        8560 roper road                           ✓
4212 Gateway Blvd NW                 4212 gateway boulevard northwest          ✓
10335 95th St NW                     10335 95 street northwest                 ✓
131 1st Ave                          131 1 avenue                              ✓

✅ All normalisation checks passed.


## 4. Load the Datasets

In [12]:
df_a = load_file(DATASET_A_PATH)
df_b = load_file(DATASET_B_PATH)

print(f"Dataset A: {df_a.shape[0]:,} rows × {df_a.shape[1]} columns")
print(f"Dataset B: {df_b.shape[0]:,} rows × {df_b.shape[1]} columns")
print()
print("Dataset A columns:", df_a.columns.tolist())
print("Dataset B columns:", df_b.columns.tolist())

Dataset A: 37 rows × 3 columns
Dataset B: 1,068 rows × 3 columns

Dataset A columns: ['Primary address', 'Property name', 'Amenities']
Dataset B columns: ['Property Address', 'Property Name', 'Amenities']


In [13]:
print("── Dataset A (first 3 rows) ──")
display(df_a.head(3))

print("── Dataset B (first 3 rows) ──")
display(df_b.head(3))

── Dataset A (first 3 rows) ──


,Primary address,Property name,Amenities
0,10047 81 Avenue Northwest,NaN,NaN
1,10425 84 Avenue Northwest,St. Anthony,NaN
2,8815 99 Street Northwest,Mill Creek West,NaN


── Dataset B (first 3 rows) ──


,Property Address,Property Name,Amenities
0,8704 51st Ave NW,NaN,"Air Conditioning, Signage"
1,9848 33 Av NW,NaN,NaN
2,7633 50th St,Meridian Place,NaN


## 5. Validate Required Columns

In [14]:
def check_col(df, col, label):
    if col not in df.columns:
        raise KeyError(
            f"Column '{col}' not found in {label}.\n"
            f"Available columns: {df.columns.tolist()}\n"
            f"Update the Configuration section at the top of this notebook."
        )
    print(f"  \u2713 {label}: '{col}'")

print("Checking columns …")
check_col(df_a, A_ADDRESS_COL,   "Dataset A")
check_col(df_a, A_AMENITIES_COL, "Dataset A")
check_col(df_b, B_ADDRESS_COL,   "Dataset B")
check_col(df_b, B_AMENITIES_COL, "Dataset B")
print("All required columns present.")

Checking columns …
  ✓ Dataset A: 'Primary address'
  ✓ Dataset A: 'Amenities'
  ✓ Dataset B: 'Property Address'
  ✓ Dataset B: 'Amenities'
All required columns present.


## 6. Build Lookup Tables from Dataset B

**For each Dataset B row, resolved canonical amenities are stored under three keys:**

| Lookup dict | Key | Used in step |
|---|---|---|
| `b_id_lookup` | Costar ID (stripped) | 1 — exact ID match |
| `b_lookup` | normalised address | 2 — address match |
| `b_name_lookup` | normalised property name | 3 — name fallback |

All three lookups are built in a single pass over Dataset B.

In [ ]:
b_id_lookup: dict[str, list[str]] = {}    # Costar ID           → canonical amenities
b_lookup: dict[str, list[str]] = {}       # normalised address  → canonical amenities
b_name_lookup: dict[str, list[str]] = {}  # normalised prop name → canonical amenities
conversion_amenity_map: dict[str, str] = {}
fuzzy_amenity_map: dict[str, str] = {}

_b_has_id_col   = B_COSTAR_ID_COL in df_b.columns
_b_has_name_col = B_PROPERTY_NAME_COL in df_b.columns

if not _b_has_id_col:
    print(f"⚠️  Column '{B_COSTAR_ID_COL}' not found in Dataset B — Costar ID matching disabled.")
if not _b_has_name_col:
    print(f"⚠️  Column '{B_PROPERTY_NAME_COL}' not found in Dataset B — property name fallback disabled.")

for _, row in df_b.iterrows():
    id_key   = str(row[B_COSTAR_ID_COL]).strip() if _b_has_id_col and not pd.isna(row[B_COSTAR_ID_COL]) else ""
    addr_key = normalise_address(row[B_ADDRESS_COL])
    name_key = normalise_name(row[B_PROPERTY_NAME_COL]) if _b_has_name_col else ""

    raw_amenities  = parse_amenities(row[B_AMENITIES_COL], B_AMENITIES_DELIMITER)
    canonical_list = []

    for raw in raw_amenities:
        canon = fuzzy_match_amenity(
            raw,
            CANONICAL_AMENITIES,
            AMENITY_FUZZY_THRESHOLD,
            conversions=AMENITY_CONVERSIONS,
            verbose=VERBOSE_FUZZY,
        )
        if canon is None:
            continue

        if raw != canon:
            if raw.lower() in AMENITY_CONVERSIONS:
                conversion_amenity_map[raw] = canon
            else:
                fuzzy_amenity_map[raw] = canon

        if canon not in canonical_list:
            canonical_list.append(canon)

    if id_key:
        if id_key not in b_id_lookup:
            b_id_lookup[id_key] = []
        for canon in canonical_list:
            if canon not in b_id_lookup[id_key]:
                b_id_lookup[id_key].append(canon)

    if addr_key:
        if addr_key not in b_lookup:
            b_lookup[addr_key] = []
        for canon in canonical_list:
            if canon not in b_lookup[addr_key]:
                b_lookup[addr_key].append(canon)

    if name_key:
        if name_key not in b_name_lookup:
            b_name_lookup[name_key] = []
        for canon in canonical_list:
            if canon not in b_name_lookup[name_key]:
                b_name_lookup[name_key].append(canon)

ids_with_amenities       = sum(1 for v in b_id_lookup.values() if v)
addresses_with_amenities = sum(1 for v in b_lookup.values() if v)
names_with_amenities     = sum(1 for v in b_name_lookup.values() if v)
print(f"ID lookup      : {len(b_id_lookup):,} unique Costar IDs     ({ids_with_amenities:,} with ≥1 amenity)")
print(f"Address lookup : {len(b_lookup):,} unique addresses     ({addresses_with_amenities:,} with ≥1 amenity)")
print(f"Name lookup    : {len(b_name_lookup):,} unique property names ({names_with_amenities:,} with ≥1 amenity)")
print(f"Explicit conversion mappings applied : {len(conversion_amenity_map):,}")
print(f"Fuzzy mappings applied               : {len(fuzzy_amenity_map):,}")

## 7. Amenity Mapping Summary

Shows how each non-exact raw value in Dataset B was resolved.
Review both tables — adjust `AMENITY_CONVERSIONS` or `AMENITY_FUZZY_THRESHOLD` if any mappings look wrong.

In [16]:
if conversion_amenity_map:
    df_conv = pd.DataFrame(
        sorted(conversion_amenity_map.items()),
        columns=["raw_value_in_dataset_b", "mapped_to_canonical"],
    )
    print(f"── Explicit conversions ({len(df_conv)}) ──")
    display(df_conv)
else:
    print("No explicit conversion rules were triggered.")

print()

if fuzzy_amenity_map:
    df_fuzzy_amenities = pd.DataFrame(
        sorted(fuzzy_amenity_map.items()),
        columns=["raw_value_in_dataset_b", "mapped_to_canonical"],
    )
    print(f"── Fuzzy matches ({len(df_fuzzy_amenities)}) ──")
    display(df_fuzzy_amenities)
else:
    print("No fuzzy amenity mappings applied — all accepted values were exact matches.")

── Explicit conversions (5) ──


,raw_value_in_dataset_b,mapped_to_canonical
0,Conferencing Facility,Conference Center
1,Convenience Store,Food Service
2,Food Court,Food Service
3,Monument Signage,Signage
4,Yard,Courtyard



── Fuzzy matches (2) ──


,raw_value_in_dataset_b,mapped_to_canonical
0,Bio-Tech/ Lab Space,Bio-Tech / Lab Space
1,Secure Storage,Storage Space


## 8. Update Dataset A's Amenities Column

**Match pipeline — first hit wins, row is skipped only if all three fail:**

1. Costar ID exact match against `b_id_lookup`
2. Normalised address exact match against `b_lookup`
3. Fuzzy WRatio address match against `b_lookup`
4. Fuzzy WRatio property name match against `b_name_lookup`

In [ ]:
df_out = df_a.copy()

matched_count         = 0
unmatched_count       = 0
skipped_count         = 0
id_match_count        = 0
addr_match_count      = 0
name_match_count      = 0
fuzzy_addr_matches    = []  # non-exact address matches (verify these)
name_fallback_matches = []  # rows matched by property name instead of address

_a_has_id_col   = A_COSTAR_ID_COL in df_out.columns
_a_has_name_col = A_PROPERTY_NAME_COL in df_out.columns

if not _a_has_id_col:
    print(f"⚠️  Column '{A_COSTAR_ID_COL}' not found in Dataset A — Costar ID matching disabled.")
if not _a_has_name_col:
    print(f"⚠️  Column '{A_PROPERTY_NAME_COL}' not found in Dataset A — property name fallback disabled.")

for idx, row in df_out.iterrows():
    matched_key  = None
    match_method = None

    # 1. Costar ID — exact match
    if _a_has_id_col:
        a_id_raw = row[A_COSTAR_ID_COL]
        if not pd.isna(a_id_raw) and str(a_id_raw).strip():
            a_id = str(a_id_raw).strip()
            if a_id in b_id_lookup:
                matched_key  = a_id
                match_method = "costar_id"

    # 2. Property address — exact then fuzzy
    if matched_key is None:
        addr_norm   = normalise_address(row[A_ADDRESS_COL])
        matched_key = fuzzy_match_address(
            addr_norm, b_lookup, ADDRESS_FUZZY_THRESHOLD, verbose=VERBOSE_FUZZY,
        )
        if matched_key is not None:
            match_method = "address"

    # 3. Property name — fuzzy fallback
    if matched_key is None and _a_has_name_col:
        a_name_raw = row[A_PROPERTY_NAME_COL]
        if not pd.isna(a_name_raw) and str(a_name_raw).strip():
            a_name_norm = normalise_name(a_name_raw)
            matched_key = fuzzy_match_name(
                a_name_norm, b_name_lookup, PROPERTY_NAME_FUZZY_THRESHOLD, verbose=VERBOSE_FUZZY,
            )
            if matched_key is not None:
                match_method = "property_name"

    if matched_key is None:
        unmatched_count += 1
        continue

    if match_method == "costar_id":
        canonical_amenities = b_id_lookup[matched_key]
    elif match_method == "address":
        canonical_amenities = b_lookup[matched_key]
        if matched_key != addr_norm:
            fuzzy_addr_matches.append({
                "dataset_a_address":         row[A_ADDRESS_COL],
                "matched_dataset_b_address": matched_key,
            })
    else:
        canonical_amenities = b_name_lookup[matched_key]
        name_fallback_matches.append({
            "dataset_a_address":  row[A_ADDRESS_COL],
            "dataset_a_name":     row[A_PROPERTY_NAME_COL],
            "matched_b_name_key": matched_key,
        })

    if not canonical_amenities:
        unmatched_count += 1
        continue

    existing  = row[A_AMENITIES_COL]
    has_value = not (pd.isna(existing) or str(existing).strip() == "")

    if has_value and not OVERWRITE_EXISTING:
        skipped_count += 1
        continue

    df_out.at[idx, A_AMENITIES_COL] = OUTPUT_DELIMITER.join(canonical_amenities)
    matched_count += 1
    if match_method == "costar_id":
        id_match_count += 1
    elif match_method == "address":
        addr_match_count += 1
    else:
        name_match_count += 1

print("Results:")
print(f"  Rows updated (total)               : {matched_count:,}")
print(f"    ↳ matched by Costar ID           : {id_match_count:,}")
print(f"    ↳ matched by address             : {addr_match_count:,}")
print(f"    ↳ matched by property name       : {name_match_count:,}")
print(f"  No match found                     : {unmatched_count:,}")
print(f"  Skipped (OVERWRITE_EXISTING=False) : {skipped_count:,}")
print(f"  Fuzzy address matches used         : {len(fuzzy_addr_matches):,}")
print(f"  Property name fallback matches     : {len(name_fallback_matches):,}")

## 9. Review Changes

In [18]:
# Side-by-side: original vs updated amenities for every changed row
changed_mask = (
    df_out[A_AMENITIES_COL].fillna("") != df_a[A_AMENITIES_COL].fillna("")
)

comparison = pd.DataFrame({
    A_ADDRESS_COL       : df_a.loc[changed_mask, A_ADDRESS_COL],
    "amenities_original": df_a.loc[changed_mask, A_AMENITIES_COL],
    "amenities_updated" : df_out.loc[changed_mask, A_AMENITIES_COL],
})

print(f"{len(comparison):,} row(s) had their amenities updated.")
display(comparison.reset_index(drop=True))

23 row(s) had their amenities updated.


,Primary address,amenities_original,amenities_updated
0,10047 81 Avenue Northwest,NaN,Signage
1,10425 84 Avenue Northwest,NaN,"Atrium, Signage"
2,11044 82 Ave,NaN,Signage
3,10507 Saskatchewan Dr,NaN,"Storage Space, Security System"
4,8215 112 St,"Property Manager on Site, 24 Hour Access, Show...","Conference Center, Fitness Center, Property Ma..."
5,8820 112 Street Northwest,NaN,"Conference Center, Fitness Center, Property Ma..."
6,10410 81 Ave,NaN,"Atrium, Signage"
7,8225 105 Street Northwest,NaN,Signage
8,10426 81 Ave,NaN,"Atrium, Signage"
9,10509 81 Avenue Northwest,NaN,Property Manager on Site


In [ ]:
# Fuzzy address matches — inspect to confirm they are the same property
if fuzzy_addr_matches:
    print(f"{len(fuzzy_addr_matches)} fuzzy address match(es) used (verify these are correct):")
    display(pd.DataFrame(fuzzy_addr_matches))
else:
    print("All address matches were exact (no fuzzy address matching was needed).")

print()

# Property name fallback matches — inspect to confirm they are the same property
if name_fallback_matches:
    print(f"{len(name_fallback_matches)} property name fallback match(es) used (verify these are correct):")
    display(pd.DataFrame(name_fallback_matches))
else:
    print("No property name fallback matches were used.")

## 10. Export Updated Dataset A

In [20]:
ext = os.path.splitext(OUTPUT_PATH)[-1].lower()

if ext == ".csv":
    df_out.to_csv(OUTPUT_PATH, index=False)
elif ext in (".xlsx", ".xls"):
    df_out.to_excel(OUTPUT_PATH, index=False)
else:
    raise ValueError(f"Unsupported output format: {ext}")

print(f"✅ Saved updated Dataset A to: {OUTPUT_PATH}")
print(f"   Shape: {df_out.shape[0]:,} rows × {df_out.shape[1]} columns")

✅ Saved updated Dataset A to: Finished amenities updated.xlsx
   Shape: 37 rows × 3 columns


## 11. Diagnostic: Unmatched Dataset A Addresses

Addresses in Dataset A that could not be linked to any Dataset B row
(even after fuzzy matching). Consider lowering `ADDRESS_FUZZY_THRESHOLD`
or checking for systematic formatting differences.

In [ ]:
_a_has_id_col_diag   = A_COSTAR_ID_COL in df_a.columns
_a_has_name_col_diag = A_PROPERTY_NAME_COL in df_a.columns

def _has_any_match(row):
    # 1. Costar ID
    if _a_has_id_col_diag:
        a_id_raw = row[A_COSTAR_ID_COL]
        if not pd.isna(a_id_raw) and str(a_id_raw).strip():
            if str(a_id_raw).strip() in b_id_lookup:
                return True
    # 2. Address
    if fuzzy_match_address(normalise_address(row[A_ADDRESS_COL]), b_lookup, ADDRESS_FUZZY_THRESHOLD) is not None:
        return True
    # 3. Property name
    if _a_has_name_col_diag:
        name_raw = row[A_PROPERTY_NAME_COL]
        if not pd.isna(name_raw) and str(name_raw).strip():
            return fuzzy_match_name(normalise_name(name_raw), b_name_lookup, PROPERTY_NAME_FUZZY_THRESHOLD) is not None
    return False

unmatched_mask = ~df_a.apply(_has_any_match, axis=1)
unmatched_rows = df_a[unmatched_mask]

cols_to_show = [A_ADDRESS_COL]
if _a_has_id_col_diag:
    cols_to_show.insert(0, A_COSTAR_ID_COL)
if _a_has_name_col_diag:
    cols_to_show.append(A_PROPERTY_NAME_COL)

print(f"{len(unmatched_rows):,} Dataset A row(s) could not be matched by ID, address, or property name:")
display(unmatched_rows[cols_to_show].drop_duplicates().reset_index(drop=True))

## 12. Diagnostic: Amenity Values That Could Not Be Matched

Raw amenity strings from Dataset B that fell below `AMENITY_FUZZY_THRESHOLD`
and were therefore discarded. Review to decide whether to:
- **Add** the value to `CANONICAL_AMENITIES` (if it’s a valid amenity)
- **Lower** `AMENITY_FUZZY_THRESHOLD` (if the match is close but scores too low)
- **Leave** it out (if it truly has no canonical equivalent)

In [22]:
from collections import Counter

still_unmatched: list[str] = []
for _, row in df_b.iterrows():
    for raw in parse_amenities(row[B_AMENITIES_COL], B_AMENITIES_DELIMITER):
        if fuzzy_match_amenity(
            raw, CANONICAL_AMENITIES, AMENITY_FUZZY_THRESHOLD,
            conversions=AMENITY_CONVERSIONS,
        ) is None:
            still_unmatched.append(raw)

freq = Counter(still_unmatched)
df_still_unmatched = (
    pd.DataFrame(
        freq.items(),
        columns=["unmatched_value", "occurrences_in_dataset_b"],
    )
    .sort_values("occurrences_in_dataset_b", ascending=False)
    .reset_index(drop=True)
)

if df_still_unmatched.empty:
    print("✅ Every amenity value in Dataset B was matched to a canonical amenity.")
else:
    print(
        f"{len(df_still_unmatched):,} distinct value(s) in Dataset B could not be matched "
        f"(threshold={AMENITY_FUZZY_THRESHOLD}):"
    )
    display(df_still_unmatched)

27 distinct value(s) in Dataset B could not be matched (threshold=80):


,unmatched_value,occurrences_in_dataset_b
0,Air Conditioning,83
1,Natural Light,34
2,Reception,27
3,Wheelchair Accessible,16
4,Day Care,15
5,Central Heating,13
6,Balcony,12
7,Fenced Lot,11
8,Basement,8
9,Outdoor Seating,6


## 13. (Optional) Interactive Threshold Tuning

Run this cell to see how the number of matched amenities and addresses
changes as you vary each threshold. Use it to find the sweet spot before
re-running the full pipeline with your chosen values.

In [23]:
print("Amenity threshold sweep (unique raw values matched vs. total raw values):")
print(f"{'Threshold':>10}  {'Matched':>8}  {'Total':>8}  {'Match %':>8}")
print("-" * 44)

all_raw_amenities = [
    raw
    for _, row in df_b.iterrows()
    for raw in parse_amenities(row[B_AMENITIES_COL], B_AMENITIES_DELIMITER)
]
unique_raw = list(set(all_raw_amenities))
total = len(unique_raw)

for t in [60, 70, 75, 80, 85, 90, 95]:
    matched = sum(
        1 for r in unique_raw
        if fuzzy_match_amenity(
            r, CANONICAL_AMENITIES, t, conversions=AMENITY_CONVERSIONS
        ) is not None
    )
    pct = matched / total * 100 if total else 0
    marker = "  ← current" if t == AMENITY_FUZZY_THRESHOLD else ""
    print(f"{t:>10}  {matched:>8,}  {total:>8,}  {pct:>7.1f}%{marker}")

print()
print("Address threshold sweep (Dataset A rows matched to at least one Dataset B entry):")
print(f"{'Threshold':>10}  {'Matched':>8}  {'Total':>8}  {'Match %':>8}")
print("-" * 44)

a_norms = df_a[A_ADDRESS_COL].apply(normalise_address).tolist()
total_a = len(a_norms)

for t in [70, 75, 80, 85, 88, 90, 92, 95]:
    matched_a = sum(
        1 for addr in a_norms
        if fuzzy_match_address(addr, b_lookup, t) is not None
    )
    pct = matched_a / total_a * 100 if total_a else 0
    marker = "  ← current" if t == ADDRESS_FUZZY_THRESHOLD else ""
    print(f"{t:>10}  {matched_a:>8,}  {total_a:>8,}  {pct:>7.1f}%{marker}")

Amenity threshold sweep (unique raw values matched vs. total raw values):
 Threshold   Matched     Total   Match %
--------------------------------------------
        60        24        51     47.1%
        70        24        51     47.1%
        75        24        51     47.1%
        80        24        51     47.1%  ← current
        85        23        51     45.1%
        90        23        51     45.1%
        95        22        51     43.1%

Address threshold sweep (Dataset A rows matched to at least one Dataset B entry):
 Threshold   Matched     Total   Match %
--------------------------------------------
        70        37        37    100.0%
        75        37        37    100.0%
        80        37        37    100.0%
        85        37        37    100.0%
        88        29        37     78.4%
        90        23        37     62.2%  ← current
        92        14        37     37.8%
        95         5        37     13.5%
